# 🎬 SIMULADOR DE PREDICCIÓN — NETFLIX CUSTOMER CHURN
## Entrenamiento y exportación de los 6 modelos de clasificación

**Conjunto de datos:** `netflix_customer_churn.csv` (5.000 clientes, 14 atributos).

**Objetivo:** reunir en un solo cuaderno los seis modelos del avance del proyecto final, entrenarlos exactamente igual que en sus cuadernos individuales (mismo preprocesamiento, misma división 80/20 y misma búsqueda de hiperparámetros) y exportarlos para usarlos en la aplicación web de **Streamlit**, donde se cambian las variables independientes y se predice la variable dependiente **Abandono** (0 = permanece, 1 = abandona).

| N.º | Modelo de clasificación |
|---|---|
| 1 | Regresión logística simple |
| 2 | Regresión logística múltiple |
| 3 | Regresión logística polinomial |
| 4 | Árbol de decisión |
| 5 | Bosques aleatorios |
| 6 | Máquina de vectores de soporte |

**Estudiantes:** Arnovis David Osorio Castillo · Diego Alfredo Pérez Corpas · Camilo Andrés Ricardo Hoyos · Simón Antonio Santana Santana
**Profesor:** Sergio Antonio Sánchez Hernández — Corporación Universitaria Antonio José de Sucre

## 1. Importación de librerías

In [ ]:
# Se importan las librerías.
import pandas as pd                 # Manipulación de datos en tablas (dataFrames)
import numpy as np                  # Operaciones numéricas
import matplotlib.pyplot as plt     # Visualización de datos
import warnings                     # Manejo de advertencias
import os, json, shutil, platform   # Archivos, exportación y versiones
import joblib                       # Guardado de los modelos entrenados
import sklearn

from sklearn import preprocessing                                   # Codificación de variables cualitativas
from sklearn.model_selection import train_test_split                # División en entrenamiento y prueba
from sklearn.model_selection import GridSearchCV, StratifiedKFold   # Búsqueda de hiperparámetros con validación cruzada
from sklearn.pipeline import make_pipeline                          # Encadenar escalado y modelo
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures  # Escalado y términos polinomiales
from sklearn.linear_model import LogisticRegression                 # Regresión logística
from sklearn.tree import DecisionTreeClassifier                     # Árbol de decisión
from sklearn.ensemble import RandomForestClassifier                 # Bosques aleatorios
from sklearn.svm import SVC                                         # Máquina de vectores de soporte

# Métricas de evaluación
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print('Librerías cargadas correctamente.')
print('Versión de Python:       ' + platform.python_version())
print('Versión de scikit-learn: ' + sklearn.__version__)

## 2. Carga del conjunto de datos

In [ ]:
# Se pide el archivo solo si todavía no está en la sesión.
if not os.path.exists('netflix_customer_churn.csv'):
    from google.colab import files
    subida = files.upload()

# Se lee el CSV y se almacena en un dataFrame de Pandas.
dataset = pd.read_csv('netflix_customer_churn.csv')
print('El conjunto de datos tiene ' + str(dataset.shape[0]) + ' renglones y ' + str(dataset.shape[1]) + ' columnas.')
dataset.head()

## 3. Preprocesamiento (idéntico en los seis cuadernos)
1. Se eliminan `customer_id` (identificador), `monthly_fee` (redundante con el tipo de suscripción) y `avg_watch_time_per_day` (derivada de las horas vistas).
2. Se renombran las columnas al español.
3. Se eliminan nulos y duplicados.
4. Se eliminan los valores atípicos con el rango intercuartílico.
5. Se codifican las variables cualitativas con `LabelEncoder`.

In [ ]:
# Se eliminan las columnas no útiles o redundantes.
dataset = dataset.drop(columns=['customer_id', 'monthly_fee', 'avg_watch_time_per_day'])

# Se renombran las columnas al español.
dataset = dataset.rename(columns={'age': 'Edad',
                                  'gender': 'Género',
                                  'subscription_type': 'Tipo_de_suscripción',
                                  'watch_hours': 'Horas_vistas',
                                  'last_login_days': 'Días_desde_último_acceso',
                                  'region': 'Región',
                                  'device': 'Dispositivo',
                                  'churned': 'Abandono',
                                  'payment_method': 'Método_de_pago',
                                  'number_of_profiles': 'Número_de_perfiles',
                                  'favorite_genre': 'Género_favorito'})

# Se eliminan los renglones con valores nulos y los duplicados.
renglones_iniciales = len(dataset)
dataset = dataset.dropna().drop_duplicates().reset_index(drop=True)
print('Renglones eliminados por nulos o duplicados: ' + str(renglones_iniciales - len(dataset)))

In [ ]:
# Función que calcula los límites superior e inferior del rango intercuartílico.
def iqrFilter(column):
    percentile25 = dataset[column].quantile(0.25)
    percentile75 = dataset[column].quantile(0.75)
    iqr = percentile75 - percentile25
    upper_limit = percentile75 + 1.5 * iqr
    lower_limit = percentile25 - 1.5 * iqr
    return upper_limit, lower_limit

columnas_numericas = ['Edad', 'Horas_vistas', 'Días_desde_último_acceso', 'Número_de_perfiles']

# Se conservan solo los datos dentro de los límites (igual que en el parcial).
temp_dataset = dataset.copy()
for columna in columnas_numericas:
    superior, inferior = iqrFilter(columna)
    temp_dataset = temp_dataset[(temp_dataset[columna] > inferior) & (temp_dataset[columna] < superior)]

print('Se eliminaron ' + str(len(dataset) - len(temp_dataset)) + ' valores atípicos.')
dataset = temp_dataset.reset_index(drop=True)
print('El conjunto de datos limpio tiene ' + str(len(dataset)) + ' renglones y ' + str(dataset.shape[1]) + ' columnas.')

In [ ]:
# Se guardan los rangos reales de las variables numéricas (los usará el simulador).
rangos = {}
for columna in columnas_numericas:
    rangos[columna] = {'min': float(dataset[columna].min()),
                       'max': float(dataset[columna].max()),
                       'mediana': float(dataset[columna].median())}

# Se codifican las variables cualitativas y se guarda el diccionario de equivalencias.
le = preprocessing.LabelEncoder()
columnas_a_codificar = ['Género', 'Tipo_de_suscripción', 'Región', 'Dispositivo', 'Método_de_pago', 'Género_favorito']

equivalencias = {}
for columna in columnas_a_codificar:
    dataset[columna] = le.fit_transform(dataset[columna])
    equivalencias[columna] = {str(k): int(v) for k, v in zip(le.classes_, le.transform(le.classes_))}

print('EQUIVALENCIAS DE LA CODIFICACIÓN:')
for columna, mapa in equivalencias.items():
    print(columna + ': ' + ', '.join(k + ' = ' + str(v) for k, v in mapa.items()))

## 4. División en entrenamiento (80 %) y prueba (20 %)

In [ ]:
X = dataset.drop(columns=['Abandono'])
y = dataset['Abandono']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Variables independientes: ' + str(list(X.columns)))
print('Ejemplos de entrenamiento: ' + str(len(X_train)))
print('Ejemplos de prueba:        ' + str(len(X_test)))

## 5. Funciones de apoyo
- **Búsqueda de hiperparámetros:** `GridSearchCV` con validación cruzada estratificada de 5 particiones y métrica F1. Entre las combinaciones casi iguales a la mejor (diferencia < 0.01) se elige la que menos sobreentrena.
- **Evaluación:** exactitud, precisión, sensibilidad, F1 y AUC ROC sobre el mismo conjunto de prueba.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def elegir_sin_sobreentrenamiento(cv_results, tolerancia=0.01):
    val = np.array(cv_results['mean_test_score'])
    brecha = np.array(cv_results['mean_train_score']) - val
    candidatos = np.where(val >= val.max() - tolerancia)[0]
    return int(candidatos[np.argmin(brecha[candidatos])])

def ajustar_modelo(modelo, parametros, X_tr):
    busqueda = GridSearchCV(modelo, parametros, cv=cv, scoring='f1',
                            return_train_score=True,
                            refit=elegir_sin_sobreentrenamiento, n_jobs=-1)
    busqueda.fit(X_tr, y_train)
    print('Mejores hiperparámetros:', busqueda.best_params_)
    return busqueda.best_estimator_, busqueda.best_params_

# Diccionarios donde se guardan los modelos, sus métricas y sus datos.
modelos, resultados, info_modelos = {}, {}, {}

def evaluar_modelo(nombre, modelo, variables):
    X_tr, X_te = X_train[variables], X_test[variables]
    y_pred = modelo.predict(X_te)
    y_prob = modelo.predict_proba(X_te)[:, 1]
    metricas = {'Exactitud (Accuracy)': accuracy_score(y_test, y_pred),
                'Precisión (Precision)': precision_score(y_test, y_pred),
                'Sensibilidad (Recall)': recall_score(y_test, y_pred),
                'Puntaje F1': f1_score(y_test, y_pred),
                'AUC ROC': roc_auc_score(y_test, y_prob),
                'Exactitud entrenamiento': modelo.score(X_tr, y_train)}
    vn, fp, fn, vp = confusion_matrix(y_test, y_pred).ravel()
    print('MÉTRICAS DEL MODELO: ' + nombre.upper())
    for metrica, valor in metricas.items():
        print('  ' + metrica.ljust(26) + str(round(valor, 4)))
    print('  Mal clasificados: ' + str(fp + fn) + ' de ' + str(len(y_test)) + ' (VN=' + str(vn) + ', FP=' + str(fp) + ', FN=' + str(fn) + ', VP=' + str(vp) + ')')
    resultados[nombre] = metricas
    return metricas

## 6. Entrenamiento de los seis modelos
### 6.1 Regresión logística simple
Se usa una sola variable independiente: la de mayor correlación absoluta con el abandono, **Días_desde_último_acceso**.

In [ ]:
nombre = 'Regresión logística simple'
variables = ['Días_desde_último_acceso']

modelo_final = make_pipeline(MinMaxScaler(), LogisticRegression(max_iter=1000))
modelo_final.fit(X_train[variables], y_train)

modelos[nombre] = modelo_final
info_modelos[nombre] = {'archivo': '1_logistica_simple.joblib', 'variables': variables, 'hiperparametros': {}}
evaluar_modelo(nombre, modelo_final, variables);

### 6.2 Regresión logística múltiple

In [ ]:
nombre = 'Regresión logística múltiple'
variables = list(X.columns)

modelo = make_pipeline(MinMaxScaler(), LogisticRegression(max_iter=1000))
parametros = {'logisticregression__C': [0.01, 0.1, 1, 10]}
modelo_final, mejores = ajustar_modelo(modelo, parametros, X_train[variables])

modelos[nombre] = modelo_final
info_modelos[nombre] = {'archivo': '2_logistica_multiple.joblib', 'variables': variables, 'hiperparametros': mejores}
evaluar_modelo(nombre, modelo_final, variables);

### 6.3 Regresión logística polinomial

In [ ]:
nombre = 'Regresión logística polinomial'
variables = list(X.columns)

modelo = make_pipeline(MinMaxScaler(),
                       PolynomialFeatures(include_bias=False),
                       LogisticRegression(max_iter=5000))
parametros = {'polynomialfeatures__degree': [2, 3],
              'logisticregression__C': [0.1, 1, 10]}
modelo_final, mejores = ajustar_modelo(modelo, parametros, X_train[variables])

modelos[nombre] = modelo_final
info_modelos[nombre] = {'archivo': '3_logistica_polinomial.joblib', 'variables': variables, 'hiperparametros': mejores}
evaluar_modelo(nombre, modelo_final, variables);

### 6.4 Árbol de decisión

In [ ]:
nombre = 'Árbol de decisión'
variables = list(X.columns)

modelo = DecisionTreeClassifier(random_state=42)
parametros = {'criterion': ['gini', 'entropy'],
              'max_depth': [3, 4, 5, 6, 8, 10, None],
              'min_samples_leaf': [1, 5, 10, 20, 40]}
modelo_final, mejores = ajustar_modelo(modelo, parametros, X_train[variables])

modelos[nombre] = modelo_final
info_modelos[nombre] = {'archivo': '4_arbol_decision.joblib', 'variables': variables, 'hiperparametros': mejores}
evaluar_modelo(nombre, modelo_final, variables);

### 6.5 Bosques aleatorios

In [ ]:
nombre = 'Bosques aleatorios'
variables = list(X.columns)

modelo = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
parametros = {'max_depth': [4, 6, 8, 10, None],
              'min_samples_leaf': [1, 5, 10, 20]}
modelo_final, mejores = ajustar_modelo(modelo, parametros, X_train[variables])

modelos[nombre] = modelo_final
info_modelos[nombre] = {'archivo': '5_bosques_aleatorios.joblib', 'variables': variables, 'hiperparametros': mejores}
evaluar_modelo(nombre, modelo_final, variables);

### 6.6 Máquina de vectores de soporte
> Esta búsqueda es la más lenta (puede tardar 1 a 2 minutos en Colab).

In [ ]:
nombre = 'Máquina de vectores de soporte'
variables = list(X.columns)

modelo = make_pipeline(MinMaxScaler(), SVC(probability=True, random_state=42))
parametros = [
    {'svc__kernel': ['linear'], 'svc__C': [0.1, 1, 10]},
    {'svc__kernel': ['rbf'], 'svc__C': [1, 10, 100], 'svc__gamma': ['scale', 1, 5]},
    {'svc__kernel': ['poly'], 'svc__degree': [2, 3], 'svc__C': [0.1, 1, 10]},
]
modelo_final, mejores = ajustar_modelo(modelo, parametros, X_train[variables])

modelos[nombre] = modelo_final
info_modelos[nombre] = {'archivo': '6_svm.joblib', 'variables': variables, 'hiperparametros': mejores}
evaluar_modelo(nombre, modelo_final, variables);

## 7. Comparación de métricas de los seis modelos
Debe coincidir con la tabla del avance del proyecto final.

In [ ]:
tabla = pd.DataFrame(resultados).T
tabla.index.name = 'Modelo'
metricas_prueba = ['Exactitud (Accuracy)', 'Precisión (Precision)', 'Sensibilidad (Recall)', 'Puntaje F1', 'AUC ROC']
tabla[metricas_prueba].round(4).style.highlight_max(axis=0, color='#c6efce').format('{:.4f}')

In [ ]:
ax = tabla[metricas_prueba].plot(kind='bar', figsize=(13, 6), width=0.8, colormap='tab10')
plt.title('COMPARACIÓN DE MÉTRICAS DE LOS MODELOS', fontdict={'family': 'DejaVu Sans', 'color': 'black', 'weight': 'bold', 'size': 13}, pad=15)
plt.ylabel('VALOR (0 a 1)', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)
plt.xlabel('')
plt.xticks(rotation=20, ha='right')
plt.ylim(0.6, 1.02)
plt.legend(loc='lower right', frameon=True, facecolor='#dddddd')
plt.suptitle('Fig. 1 Métricas de los seis modelos sobre el mismo conjunto de prueba (953 clientes).', fontproperties={'family': 'DejaVu Sans', 'size': 11}, y=-0.02)
plt.tight_layout()
plt.show()

## 8. Exportación de los modelos para Streamlit
Se crea la carpeta `modelos/` con:
- un archivo `.joblib` por modelo entrenado,
- `metadata.json` (variables, equivalencias de la codificación, rangos, métricas e hiperparámetros),
- `prueba.csv` (los 953 clientes de prueba, para cargar ejemplos reales en el simulador).

También se genera `requirements.txt` con las **mismas versiones** de las librerías usadas aquí, para que Streamlit Cloud cargue los modelos sin problemas de compatibilidad.

In [ ]:
carpeta = 'modelos'
os.makedirs(carpeta, exist_ok=True)

# Se guarda cada modelo entrenado.
for nombre, modelo in modelos.items():
    joblib.dump(modelo, os.path.join(carpeta, info_modelos[nombre]['archivo']), compress=3)
    print('Guardado: ' + info_modelos[nombre]['archivo'])

# Se guardan los 953 clientes de prueba con su etiqueta real.
prueba = X_test.copy()
prueba['Abandono'] = y_test.values
prueba.to_csv(os.path.join(carpeta, 'prueba.csv'), index=False)

# Se guarda la información que necesita el simulador.
def a_python(valor):
    return valor.item() if hasattr(valor, 'item') else valor

metadata = {
    'variables': list(X.columns),
    'variable_objetivo': 'Abandono',
    'columnas_numericas': columnas_numericas,
    'columnas_categoricas': columnas_a_codificar,
    'equivalencias': equivalencias,
    'rangos': rangos,
    'modelos': {nombre: {'archivo': info['archivo'],
                         'variables': info['variables'],
                         'hiperparametros': {k: a_python(v) for k, v in info['hiperparametros'].items()},
                         'metricas': {k: round(float(v), 4) for k, v in resultados[nombre].items()}}
                for nombre, info in info_modelos.items()},
    'datos': {'renglones_limpios': int(len(dataset)), 'entrenamiento': int(len(X_train)), 'prueba': int(len(X_test))},
    'versiones': {'python': platform.python_version(), 'scikit-learn': sklearn.__version__,
                  'pandas': pd.__version__, 'numpy': np.__version__, 'joblib': joblib.__version__},
}
with open(os.path.join(carpeta, 'metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# requirements.txt con las versiones exactas de esta sesión.
with open('requirements.txt', 'w') as f:
    f.write('streamlit\n')
    f.write('scikit-learn==' + sklearn.__version__ + '\n')
    f.write('pandas==' + pd.__version__ + '\n')
    f.write('numpy==' + np.__version__ + '\n')
    f.write('joblib==' + joblib.__version__ + '\n')
    f.write('plotly\n')

print()
print('Exportación completa. Contenido de requirements.txt:')
print(open('requirements.txt').read())

## 9. Descarga del paquete
Se comprime la carpeta `modelos/` junto con `requirements.txt` y se descarga. Su contenido se sube al repositorio de GitHub de la app de Streamlit.

In [ ]:
os.makedirs('paquete_streamlit', exist_ok=True)
shutil.copytree(carpeta, os.path.join('paquete_streamlit', carpeta), dirs_exist_ok=True)
shutil.copy('requirements.txt', 'paquete_streamlit')
shutil.copy('netflix_customer_churn.csv', 'paquete_streamlit')
shutil.make_archive('modelos_streamlit', 'zip', 'paquete_streamlit')
print('Archivo generado: modelos_streamlit.zip')

try:
    from google.colab import files
    files.download('modelos_streamlit.zip')
except ImportError:
    print('(Fuera de Colab: el zip queda en la carpeta actual.)')